# Data Retrieval Reddit

| Field | Details |
|---|---|
| **Time window** | 5 Jul 2024 – 4 Nov 2024 |
| **Source** | Reddit — JSONL files (Pushshift / Academic Data Access) |
| **Collection** | Pre-downloaded before Reddit API shutdown by external contributor; not collected by this team |
| **Subreddits** | r/conservative · r/democrats · r/republican · r/worldnews · r/politics · r/liberal · r/trump |
| **Method** | Merge all per-subreddit JSONL files; filter inline to date window and election keywords (same keyword set as Bluesky) |
| **Keywords** | trump · harris · kamala · biden · vance · walz · election · vote · ballot · debate · maga · gop · dnc · rnc · swing state · battleground · project2025 · … |
| **Saved to** | · `Data/1_Bronze/Reddit/reddit_comments_filtered.parquet` · `Data/1_Bronze/Reddit/reddit_posts_filtered.parquet` |

### reddit_comments_filtered.parquet

De gefilterde versie bevat dezelfde kolommen als de raw versie, maar enkel rijen binnen het datumvenster die minstens één verkiezingskeyword bevatten.

| Column | Type | Description |
|---|---|---|
| `subreddit` | str | Subreddit name (conservative / democrats / republican / worldnews / politics / liberal / trump) |
| `created_utc` | datetime (UTC) | Comment timestamp (derived from Unix epoch via `created_utc` field) |
| `id` | str | Reddit comment ID |
| `author` | str | Reddit username of the commenter |
| `body` | str | Comment text |
| `score` | int | Net upvote score |
| `ups` | int | Raw upvote count |
| `link_id` | str | Root post ID (`t3_XXXXX`) — links comment back to its parent post |
| `parent_id` | str | Direct parent ID (`t1_XXXXX` = parent comment · `t3_XXXXX` = direct reply to post) |
| `permalink` | str | Relative URL path to the comment |
| `controversiality` | int | 1 if marked controversial by Reddit, otherwise 0 |
| `stickied` | bool | Whether the comment was stickied by a moderator |

### reddit_posts_filtered.parquet

De gefilterde versie bevat dezelfde kolommen als de raw versie, maar enkel rijen binnen het datumvenster waarbij title of selftext minstens één verkiezingskeyword bevat.

| Column | Type | Description |
|---|---|---|
| `subreddit` | str | Subreddit name |
| `created_utc` | datetime (UTC) | Post timestamp (derived from Unix epoch via `created_utc` field) |
| `id` | str | Reddit post ID |
| `author` | str | Reddit username of the poster |
| `title` | str | Post title |
| `selftext` | str | Post body text (empty for link posts) |
| `score` | int | Net upvote score |
| `ups` | int | Raw upvote count |
| `upvote_ratio` | float | Fraction of upvotes out of total votes (0–1) |
| `num_comments` | int | Total number of comments on the post |
| `url` | str | Linked URL (for link posts) |
| `domain` | str | Domain of the linked URL |
| `permalink` | str | Relative URL path to the post |
| `is_self` | bool | True if text post, False if link post |
| `over_18` | bool | Whether the post is marked NSFW |

In [3]:
import json
import pandas as pd
from pathlib import Path

BRONZE = Path('../../Data/1_Bronze/Reddit')
print('Setup complete.')
print(f'Bronze folder: {BRONZE.resolve()}')

Setup complete.
Bronze folder: C:\Users\verme_hzys4y0\UGent\2025-2026\Social media and webanalytics\Data\1_Bronze\Reddit


In [2]:
jsonl_files = sorted(BRONZE.glob('*.jsonl'))
print(f'Found {len(jsonl_files)} .jsonl files:')
for f in jsonl_files:
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<45}  {size_mb:.1f} MB')

Found 14 .jsonl files:
  r_conservative_comments.jsonl                  1136.2 MB
  r_conservative_posts.jsonl                     157.9 MB
  r_democrats_comments.jsonl                     767.2 MB
  r_democrats_posts.jsonl                        88.2 MB
  r_liberal_comments.jsonl                       33.4 MB
  r_liberal_posts.jsonl                          6.3 MB
  r_politics_comments.jsonl                      3783.9 MB
  r_politics_posts.jsonl                         288.0 MB
  r_republican_comments.jsonl                    166.1 MB
  r_republican_posts.jsonl                       36.7 MB
  r_trump_comments.jsonl                         320.2 MB
  r_trump_posts.jsonl                            80.7 MB
  r_worldnews_comments.jsonl                     3228.1 MB
  r_worldnews_posts.jsonl                        147.8 MB


In [3]:
import pyarrow as pa
import pyarrow.parquet as pq

# Output folders — one parquet file per subreddit inside each folder
comments_dir = BRONZE / 'comments'
posts_dir    = BRONZE / 'posts'
comments_dir.mkdir(exist_ok=True)
posts_dir.mkdir(exist_ok=True)

CHUNKSIZE = 50_000

# Numeric promotion order: higher rank wins when types conflict
_TYPE_RANK = {pa.bool_(): 0, pa.int32(): 1, pa.int64(): 2,
              pa.float32(): 3, pa.float64(): 4, pa.large_utf8(): 99}

def _wider(t1, t2):
    if t1 == t2:
        return t1
    r1 = _TYPE_RANK.get(t1, -1)
    r2 = _TYPE_RANK.get(t2, -1)
    if r1 >= 0 and r2 >= 0:
        return t1 if r1 >= r2 else t2
    return pa.large_utf8()  # fallback: convert to string

def merge_schemas(s1, s2):
    """Merge two Arrow schemas field by field, promoting conflicting types."""
    fields = {f.name: f.type for f in s1}
    for f in s2:
        fields[f.name] = _wider(fields[f.name], f.type) if f.name in fields else f.type
    return pa.schema([pa.field(n, t) for n, t in fields.items()])

def drop_nested_columns(df):
    """Drop columns containing dicts or lists — not parquet-compatible."""
    nested = [c for c in df.columns
              if df[c].dropna().apply(lambda x: isinstance(x, (dict, list))).any()]
    return df.drop(columns=nested)

def iter_chunks(f):
    """Yield cleaned pandas chunks from a JSONL file."""
    for chunk in pd.read_json(f, lines=True, chunksize=CHUNKSIZE):
        chunk['subreddit'] = chunk['subreddit'].str.lower()
        if 'created_utc' in chunk.columns:
            chunk['created_utc'] = pd.to_datetime(chunk['created_utc'], unit='s', utc=True)
        yield drop_nested_columns(chunk)

def build_unified_schema(f):
    """Pass 1: scan all chunks to build a unified Arrow schema."""
    schema = None
    for chunk in iter_chunks(f):
        tbl = pa.Table.from_pandas(chunk, preserve_index=False)
        schema = tbl.schema if schema is None else merge_schemas(schema, tbl.schema)
    return schema

def align_table(table, schema):
    """Align a table to the target schema: reorder cols, add missing as null, cast types."""
    arrays = []
    for field in schema:
        if field.name in table.schema.names:
            col = table.column(field.name)
            if col.type != field.type:
                try:
                    col = col.cast(field.type, safe=False)
                except Exception:
                    col = pa.nulls(len(table), type=field.type)
            arrays.append(col)
        else:
            arrays.append(pa.nulls(len(table), type=field.type))
    return pa.table(dict(zip(schema.names, arrays)), schema=schema)

def convert_file(f, out_dir):
    out_path = out_dir / f.name.replace('.jsonl', '.parquet')

    print(f'  {f.name} — building schema (pass 1/2)...', end='\r')
    schema = build_unified_schema(f)

    writer = pq.ParquetWriter(out_path, schema)
    total = 0
    for chunk in iter_chunks(f):
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        table = align_table(table, schema)
        writer.write_table(table)
        total += len(chunk)
    writer.close()
    print(f'  {f.name:<45}  {total:>7,} records  →  {out_path.name}')

print('Converting comments...')
for f in sorted(BRONZE.glob('*_comments.jsonl')):
    convert_file(f, comments_dir)

print('\nConverting posts...')
for f in sorted(BRONZE.glob('*_posts.jsonl')):
    convert_file(f, posts_dir)

print('\nDone.')
print(f'Comments folder: {comments_dir}')
print(f'Posts folder   : {posts_dir}')

Converting comments...
  r_conservative_comments.jsonl                  693,340 records  →  r_conservative_comments.parquet
  r_democrats_comments.jsonl                     444,280 records  →  r_democrats_comments.parquet
  r_liberal_comments.jsonl                        19,652 records  →  r_liberal_comments.parquet
  r_politics_comments.jsonl                      2,104,000 records  →  r_politics_comments.parquet
  r_republican_comments.jsonl                     93,495 records  →  r_republican_comments.parquet
  r_trump_comments.jsonl                         188,947 records  →  r_trump_comments.parquet
  r_worldnews_comments.jsonl                     1,847,724 records  →  r_worldnews_comments.parquet

Converting posts...
  r_conservative_posts.jsonl                      41,790 records  →  r_conservative_posts.parquet
  r_democrats_posts.jsonl                         20,756 records  →  r_democrats_posts.parquet
  r_liberal_posts.jsonl                            1,887 records  →  r_liber

In [4]:
def merge_parquet_dir(input_dir, out_path):
    """Merge all parquet files in input_dir into a single parquet file at out_path."""
    files = sorted(input_dir.glob('*.parquet'))
    if not files:
        print(f'  No parquet files found in {input_dir}')
        return

    # Pass 1: build unified schema across all files
    schema = None
    for f in files:
        s = pq.read_schema(f)
        # Strip pandas metadata to avoid conflicts; keep only field names/types
        s = pa.schema([pa.field(f.name, f.type) for f in s])
        schema = s if schema is None else merge_schemas(schema, s)

    # Pass 2: stream row groups from each file and write to combined file
    writer = pq.ParquetWriter(out_path, schema)
    total = 0
    for f in files:
        pf = pq.ParquetFile(f)
        n = 0
        for batch in pf.iter_batches(batch_size=100_000):
            table = pa.Table.from_batches([batch])
            table = align_table(table, schema)
            writer.write_table(table)
            n += len(table)
        total += n
        print(f'  {f.name:<45}  {n:>7,} records')
    writer.close()
    print(f'  → {out_path.name}  ({total:,} records total)\n')


print('Merging comments...')
merge_parquet_dir(comments_dir, comments_dir / 'reddit_comments_raw.parquet')

print('Merging posts...')
merge_parquet_dir(posts_dir, posts_dir / 'reddit_posts_raw.parquet')

print('Done.')
print(f'  {comments_dir / "reddit_comments_raw.parquet"}')
print(f'  {posts_dir / "reddit_posts_raw.parquet"}')

Merging comments...
  r_conservative_comments.parquet                693,340 records
  r_democrats_comments.parquet                   444,280 records
  r_liberal_comments.parquet                      19,652 records
  r_politics_comments.parquet                    2,104,000 records
  r_republican_comments.parquet                   93,495 records
  r_trump_comments.parquet                       188,947 records
  r_worldnews_comments.parquet                   1,847,724 records
  → reddit_comments_raw.parquet  (5,391,438 records total)

Merging posts...
  r_conservative_posts.parquet                    41,790 records
  r_democrats_posts.parquet                       20,756 records
  r_liberal_posts.parquet                          1,887 records
  r_politics_posts.parquet                        69,963 records
  r_republican_posts.parquet                       9,707 records
  r_trump_posts.parquet                           19,111 records
  r_worldnews_posts.parquet                       36,8

In [7]:
import pandas as pd
from pathlib import Path

BRONZE = Path('../Data/1_Bronze/Reddit')

# -- 1. Date window
DATE_FROM = pd.Timestamp('2024-07-05', tz='UTC')
DATE_TO   = pd.Timestamp('2024-11-04 23:59:59', tz='UTC')

# -- 2. Election keywords (identical to Bluesky)
ELECTION_KEYWORDS = [
    # Candidates & running mates
    'trump', 'donald trump', 'harris', 'kamala', 'kamala harris',
    'biden', 'joe biden', 'vance', 'jd vance', 'walz', 'tim walz',
    # Election process
    'election', 'election2024', 'us election', 'electionday',
    'vote', 'vote2024', 'voting', 'voter', 'voterregistration',
    'ballot', 'earlyvoting', 'mail-in voting', 'mailinvoting',
    'poll', 'polls', 'electoral', 'swing state', 'battleground',
    'electionintegrity', 'america decides', 'decision2024',
    # Campaign & parties
    'campaign', 'debate', 'presidential debate', 'vp debate',
    'trumpdebate', 'trumpharrisdebate', 'debatenight',
    'president', 'presidential',
    'democrat', 'democratic', 'democrats',
    'republican', 'republicans',
    'maga', 'maga2024', 'america first', 'americafirst',
    'gop', 'dnc', 'rnc',
    'project2025',
    # Conventions & events
    'convention', 'rnc2024', 'dnc2024', 'republican convention',
    'democratic convention',
    'trump rally', 'trumprally',
    'inauguration', 'primary',
    # Biden exit
    'biden drops out', 'bidendropsout', 'biden out', 'biden withdraws',
    # Slogans
    'vote blue', 'voteblue', 'vote trump', 'votetrump',
    'vote harris', 'voteharris', 'vote kamala',
    'we are not going back', 'harriswalz', 'trumpvance',
]

def has_keyword(text):
    if not isinstance(text, str):
        return False
    t = text.lower()
    return any(kw in t for kw in ELECTION_KEYWORDS)

# -- 3. Load, filter & save comments
print('Loading comments...')
comments = pd.read_parquet(BRONZE / 'reddit_comments_raw.parquet')
print(f'  Raw: {len(comments):,} rows')

mask_date = (comments['created_utc'] >= DATE_FROM) & (comments['created_utc'] <= DATE_TO)
comments = comments[mask_date]
print(f'  After date filter: {len(comments):,} rows')

mask_kw = comments['body'].apply(has_keyword)
comments = comments[mask_kw]
print(f'  After keyword filter: {len(comments):,} rows')

out_comments = BRONZE / 'reddit_comments_filtered.parquet'
comments.to_parquet(out_comments, index=False, engine='pyarrow')
print(f'  Saved -> {str(out_comments)}')
del comments

# -- 4. Load, filter & save posts
print()
print('Loading posts...')
posts = pd.read_parquet(BRONZE / 'reddit_posts_raw.parquet')
print(f'  Raw: {len(posts):,} rows')

mask_date = (posts['created_utc'] >= DATE_FROM) & (posts['created_utc'] <= DATE_TO)
posts = posts[mask_date]
print(f'  After date filter: {len(posts):,} rows')

mask_kw = posts['title'].apply(has_keyword) | posts['selftext'].apply(has_keyword)
posts = posts[mask_kw]
print(f'  After keyword filter: {len(posts):,} rows')

out_posts = BRONZE / 'reddit_posts_filtered.parquet'
posts.to_parquet(out_posts, index=False, engine='pyarrow')
print(f'  Saved -> {str(out_posts)}')
del posts

print()
print('Done.')


Loading comments...
  Raw: 5,391,438 rows
  After date filter: 5,391,438 rows
  After keyword filter: 1,400,420 rows
  Saved -> ..\Data\1_Bronze\Reddit\reddit_comments_filtered.parquet

Loading posts...
  Raw: 200,068 rows
  After date filter: 199,599 rows
  After keyword filter: 111,003 rows
  Saved -> ..\Data\1_Bronze\Reddit\reddit_posts_filtered.parquet

Done.
